### External

In [ ]:
# ================================
# CHUNK 1/10
# Importación de dependencias para adquisición de datos externos
# ================================
# Este bloque importa todas las bibliotecas necesarias para:
# 1. Descargar datasets desde Kaggle utilizando kagglehub.
# 2. Manipular rutas del sistema de archivos de forma multiplataforma (pathlib).
# 3. Copiar estructuras de directorios y archivos (shutil).
# 4. Manipular archivos comprimidos en caso de ser necesario (zipfile).
# 5. Descargar datasets desde HuggingFace Hub (datasets.load_dataset).
#
# Consideraciones técnicas:
# - kagglehub permite descargar datasets sin necesidad de manejar manualmente
#   autenticación vía API tradicional de Kaggle (aunque depende del entorno).
# - pathlib.Path se utiliza en lugar de strings para rutas porque:
#     * Es más robusto.
#     * Es multiplataforma.
#     * Permite composición segura de rutas.
# - load_dataset permite cargar datasets directamente en memoria como objetos
#   tipo Dataset, optimizados con Apache Arrow.
#
# Nota: La advertencia de tqdm/IProgress está relacionada con widgets en Jupyter
# y no afecta la ejecución del pipeline.

import kagglehub
import os
import zipfile
import os
from pathlib import Path
import shutil
from datasets import load_dataset

/Users/msgarcia/Desktop/School/01_DL/autoencoder-project/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ================================
# CHUNK 2/10
# Descarga y copia del dataset desde Kaggle hacia la carpeta external
# ================================
# Objetivo:
# - Descargar el dataset "human-faces-dataset" desde Kaggle.
# - Copiarlo dentro de la estructura del proyecto en data/external.
#
# Flujo lógico:
# 1. Se descarga el dataset usando kagglehub.
# 2. Se define la carpeta destino dentro del proyecto.
# 3. Se asegura que la carpeta destino exista.
# 4. Se copia el contenido descargado a la carpeta external.
#
# Estructura del proyecto asumida:
# project_root/
# └── data/
#     └── external/
#
# dirs_exist_ok=True permite sobreescritura sin lanzar error,
# lo cual es útil para ejecuciones repetidas en notebooks.

# Descargar dataset desde Kaggle
path = kagglehub.dataset_download("kaustubhdhote/human-faces-dataset")

# Definir ruta destino relativa al notebook (../data/external)
destino = Path("../data/external")

# Crear carpeta si no existe
destino.mkdir(parents=True, exist_ok=True)

# Copiar archivos descargados al proyecto
shutil.copytree(path, destino, dirs_exist_ok=True)

print("Archivos copiados correctamente")

Archivos copiados correctamente


In [ ]:
# ================================
# CHUNK 3/10
# Descarga del dataset CelebA desde HuggingFace
# ================================
# Objetivo:
# - Descargar el dataset CelebA (caras reales) desde HuggingFace.
# - Preparar carpeta destino dentro de data/external/celeba.
#
# Consideraciones:
# - load_dataset descarga y cachea el dataset automáticamente.
# - split="train" indica que solo se descarga el split de entrenamiento.
# - CelebA contiene 202,599 imágenes de rostros reales.
#
# Este dataset se utilizará posteriormente para reforzar la clase "real".

PROJECT_ROOT = Path().resolve().parent

CELEBA_PATH = PROJECT_ROOT / "data" / "external" / "celeba"
CELEBA_PATH.mkdir(parents=True, exist_ok=True)

# Cargar dataset desde HuggingFace Hub
celeba_dataset = load_dataset("nielsr/CelebA-faces", split="train")

print(f"CelebA size: {len(celeba_dataset)}")

CelebA size: 202599


### Raw

In [ ]:
# ================================
# CHUNK 4/10
# Importaciones para procesamiento RAW
# ================================
# Este bloque prepara dependencias necesarias para:
# - Validar imágenes (PIL.Image).
# - Manipular rutas (Path).
# - Copiar archivos (shutil).
#
# PIL.Image.verify() se utilizará para detectar imágenes corruptas.
# Esto es crítico en pipelines reales, ya que imágenes dañadas pueden:
# - Romper el entrenamiento.
# - Generar errores silenciosos.
# - Introducir ruido no controlado.

from pathlib import Path
from PIL import Image
import shutil

In [ ]:
# ================================
# CHUNK 5/10
# Construcción del dataset RAW validado
# ================================
# Objetivo:
# - Tomar imágenes desde data/external.
# - Verificar integridad.
# - Copiar solo imágenes válidas a data/raw.
#
# Etapas:
# 1. Definir rutas fuente y destino.
# 2. Crear carpetas destino.
# 3. Implementar función process_images para validación.
# 4. Generar reporte estadístico básico.
#
# Importancia:
# La validación evita incluir imágenes corruptas que podrían causar
# errores durante el entrenamiento del modelo.

# Definir rutas principales
PROJECT_ROOT = Path().resolve().parent
EXTERNAL_PATH = PROJECT_ROOT / "data" / "external" / "human faces dataset"
RAW_PATH = PROJECT_ROOT / "data" / "raw"

AI_SRC = EXTERNAL_PATH / "ai-generated images"
REAL_SRC = EXTERNAL_PATH / "real images"

AI_DST = RAW_PATH / "ai"
REAL_DST = RAW_PATH / "real"

AI_DST.mkdir(parents=True, exist_ok=True)
REAL_DST.mkdir(parents=True, exist_ok=True)

# Función de validación y copia
def process_images(src_folder, dst_folder):
    total = 0
    valid = 0
    corrupted = 0
    
    for img_path in src_folder.glob("*"):
        total += 1
        
        try:
            with Image.open(img_path) as img:
                img.verify()  # Validación estructural
            
            shutil.copy(img_path, dst_folder / img_path.name)
            valid += 1
            
        except Exception:
            corrupted += 1
    
    return total, valid, corrupted

# Ejecutar procesamiento
ai_total, ai_valid, ai_corrupted = process_images(AI_SRC, AI_DST)
real_total, real_valid, real_corrupted = process_images(REAL_SRC, REAL_DST)

print("===== DATASET REPORT =====")
print(f"AI Images     → Total: {ai_total}, Válidas: {ai_valid}, Corruptas: {ai_corrupted}")
print(f"Real Images   → Total: {real_total}, Válidas: {real_valid}, Corruptas: {real_corrupted}")
print(f"Total limpio en RAW: {ai_valid + real_valid}")

===== DATASET REPORT =====
AI Images     → Total: 4630, Válidas: 4630, Corruptas: 0
Real Images   → Total: 5000, Válidas: 5000, Corruptas: 0
Total limpio en RAW: 9630


In [ ]:
# ================================
# CHUNK 6/10
# Análisis de balance de clases
# ================================
# Este bloque calcula la proporción relativa de imágenes AI vs Real.
#
# Importancia:
# - Detectar desbalance temprano.
# - Evaluar necesidad de técnicas de balanceo posteriores.
#
# Fórmula:
# porcentaje = (cantidad_clase / total) * 100

print("\nBalance de clases:")
print(f"% AI   : {ai_valid / (ai_valid + real_valid) * 100:.2f}%")
print(f"% Real : {real_valid / (ai_valid + real_valid) * 100:.2f}%")


Balance de clases:
% AI   : 48.08%
% Real : 51.92%


In [ ]:
# ================================
# CHUNK 7/10
# Incorporación de CelebA al dataset RAW como clase real adicional
# ================================
# Objetivo:
# - Aumentar la diversidad de la clase "real".
# - Limitar el número de imágenes para evitar desbalance extremo.
#
# max_images permite controlar el tamaño del subconjunto utilizado.
#
# Conversión a RGB garantiza consistencia de canales.

CELEBA_RAW = RAW_PATH / "celeba_real"
CELEBA_RAW.mkdir(parents=True, exist_ok=True)

max_images = 50000

for i, example in enumerate(celeba_dataset):
    if i >= max_images:
        break
    
    img = example["image"].convert("RGB")
    img.save(CELEBA_RAW / f"celeba_{i}.jpg")

print(f"Saved {max_images} CelebA images to RAW")

Saved 50000 CelebA images to RAW


### Interim

In [ ]:
# ================================
# CHUNK 8/10
# Importaciones para construcción del dataset INTERIM
# ================================
# Se preparan dependencias necesarias para:
# - Aleatorización reproducible (random).
# - Copia de archivos.
# - Manejo estructurado de rutas.

import os
import random
import shutil
from pathlib import Path

In [ ]:
# ================================
# CHUNK 9/10
# División Train / Validation / Test
# ================================
# Objetivo:
# - Separar dataset RAW en:
#     * 70% Train
#     * 15% Validation
#     * 15% Test
#
# Reproducibilidad:
# - Se fija semilla 42.
#
# Función split_class_images_multi:
# - Permite combinar múltiples fuentes en una sola clase.
# - Mezcla aleatoriamente.
# - Copia archivos a carpetas correspondientes.

random.seed(42)

PROJECT_ROOT = Path().resolve().parent
RAW_PATH = PROJECT_ROOT / "data" / "raw"
INTERIM_PATH = PROJECT_ROOT / "data" / "interim"

train_split = 0.7
val_split = 0.15
test_split = 0.15

for split in ["train", "val", "test"]:
    for cls in ["ai", "real"]:
        (INTERIM_PATH / split / cls).mkdir(parents=True, exist_ok=True)

def split_class_images_multi(source_dirs, class_name):
    all_images = []
    
    for source_dir in source_dirs:
        all_images.extend(list(source_dir.glob("*")))
    
    random.shuffle(all_images)
    
    total = len(all_images)
    train_size = int(total * train_split)
    val_size = int(total * val_split)
    
    train_imgs = all_images[:train_size]
    val_imgs = all_images[train_size:train_size + val_size]
    test_imgs = all_images[train_size + val_size:]
    
    splits = {
        "train": train_imgs,
        "val": val_imgs,
        "test": test_imgs
    }
    
    for split_name, img_list in splits.items():
        for img_path in img_list:
            shutil.copy(
                img_path,
                INTERIM_PATH / split_name / class_name / img_path.name
            )
    
    return total, len(train_imgs), len(val_imgs), len(test_imgs)

ai_stats = split_class_images_multi([RAW_PATH / "ai"], "ai")

real_stats = split_class_images_multi(
    [RAW_PATH / "real", RAW_PATH / "celeba_real"],
    "real"
)

print("===== SPLIT REPORT =====")
print(f"AI    → Total: {ai_stats[0]} | Train: {ai_stats[1]} | Val: {ai_stats[2]} | Test: {ai_stats[3]}")
print(f"REAL  → Total: {real_stats[0]} | Train: {real_stats[1]} | Val: {real_stats[2]} | Test: {real_stats[3]}")

===== SPLIT REPORT =====
AI    → Total: 4630 | Train: 3241 | Val: 694 | Test: 695
REAL  → Total: 55000 | Train: 38500 | Val: 8250 | Test: 8250


In [ ]:
# ================================
# CHUNK 10/10
# Generación de Manifest estructurado
# ================================
# Objetivo:
# - Crear un archivo CSV con:
#     * split
#     * clase
#     * label numérico
#     * ruta completa
#
# Ventajas:
# - Reproducibilidad.
# - Facilita carga posterior con DataLoader.
# - Evita reconstrucción con glob().
#
# label_map:
# - real = 0
# - ai = 1
#
# El archivo se guarda como:
# data/interim/manifest_interim.csv

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
INTERIM_PATH = PROJECT_ROOT / "data" / "interim"

rows = []
label_map = {"real": 0, "ai": 1}

for split in ["train", "val", "test"]:
    for cls in ["real", "ai"]:
        folder = INTERIM_PATH / split / cls
        for p in folder.glob("*"):
            rows.append({
                "split": split,
                "class": cls,
                "label": label_map[cls],
                "path": str(p)
            })

df_manifest = pd.DataFrame(rows)
manifest_path = INTERIM_PATH / "manifest_interim.csv"
df_manifest.to_csv(manifest_path, index=False)

print("Manifest guardado en:", manifest_path)
print("Filas:", len(df_manifest))
print(df_manifest.head())

Manifest guardado en: /Users/msgarcia/Desktop/School/01_DL/autoencoder-project/data/interim/manifest_interim.csv
Filas: 60674
   split class  label                                               path
0  train  real      0  /Users/msgarcia/Desktop/School/01_DL/autoencod...
1  train  real      0  /Users/msgarcia/Desktop/School/01_DL/autoencod...
2  train  real      0  /Users/msgarcia/Desktop/School/01_DL/autoencod...
3  train  real      0  /Users/msgarcia/Desktop/School/01_DL/autoencod...
4  train  real      0  /Users/msgarcia/Desktop/School/01_DL/autoencod...
